In [2]:
# ══════════════════════════════════════════════════════════════
# 0. IMPORTS
# ══════════════════════════════════════════════════════════════
import gc
import logging
import warnings
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from scipy import stats
from sklearn.metrics import auc as sklearn_auc
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from lime import lime_image
 
try:
    import timm
except ImportError:
    raise ImportError("Install timm dulu: pip install timm")
 
warnings.filterwarnings("ignore")
 
# ══════════════════════════════════════════════════════════════
# 1. CONFIG
# ══════════════════════════════════════════════════════════════
CFG = {
    "model_name":       "XceptionNet",
    "checkpoint":       "/kaggle/input/datasets/wesleypeyfendo/...",
    "test_dir":         "/kaggle/input/datasets/...",
    "output_dir":       "xai_outputs",
    "img_size":         299,
    "norm_mean":        [0.5, 0.5, 0.5],
    "norm_std":         [0.5, 0.5, 0.5],
    "n_samples":        300,
    "random_seed":      42,
    "lime_num_samples": 1000,
    "deletion_steps":   50,
}
 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(CFG["random_seed"])
torch.manual_seed(CFG["random_seed"])
OUT = Path(CFG["output_dir"])
OUT.mkdir(parents=True, exist_ok=True)
 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(OUT / "xai_xceptionnet.log"),
    ],
)
log = logging.getLogger(__name__)
log.info(f"Device : {DEVICE}")
log.info(f"Model  : {CFG['model_name']}")
 
 
# ══════════════════════════════════════════════════════════════
# 2. DATASET LOADER
# ══════════════════════════════════════════════════════════════
 
def build_dataframe(test_dir: str) -> pd.DataFrame:
    root = Path(test_dir)
    LABEL_MAP = {"real": 0, "original": 0, "fake": 1,
                 "deepfake": 1, "deepfakes": 1, "manipulated": 1}
    records = []
    for subfolder in sorted(root.iterdir()):
        if not subfolder.is_dir():
            continue
        fname = subfolder.name.lower()
        if fname not in LABEL_MAP:
            log.warning(f"Folder '{subfolder.name}' tidak dikenali, dilewati.")
            continue
        label = LABEL_MAP[fname]
        imgs  = []
        for ext in ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]:
            imgs.extend(subfolder.glob(ext))
            imgs.extend(subfolder.glob(ext.upper()))
 
        for p in imgs:
            records.append({
                "abs_path":          str(p),
                "binary_label":      label,
                "manipulation_type": "real" if label == 0 else "fake",
                "filename":          p.name,
            })
        log.info(f"  '{subfolder.name}' → label={label} | {len(imgs)} gambar")
 
    df = pd.DataFrame(records)
    log.info(f"Total: {len(df)} | Real: {(df['binary_label']==0).sum()} | Fake: {(df['binary_label']==1).sum()}")
    return df
 
 
def sample_balanced(df: pd.DataFrame) -> pd.DataFrame:
    n_each = min(CFG["n_samples"] // 2,
                 (df["binary_label"] == 0).sum(),
                 (df["binary_label"] == 1).sum())
    sampled = pd.concat([
        df[df["binary_label"] == 0].sample(n_each, random_state=CFG["random_seed"]),
        df[df["binary_label"] == 1].sample(n_each, random_state=CFG["random_seed"]),
    ]).reset_index(drop=True)
    log.info(f"Sampling: {len(sampled)} gambar ({n_each} real + {n_each} fake)")
    return sampled
 
 
# ══════════════════════════════════════════════════════════════
# 3. MODEL
# ══════════════════════════════════════════════════════════════
 
def build_model() -> nn.Module:
    base = timm.create_model("xception", pretrained=False, num_classes=0)
    for m in base.modules():
        if isinstance(m, nn.ReLU):
            m.inplace = False
 
    class XceptionNet(nn.Module):
        def __init__(self, backbone, in_feat):
            super().__init__()
            self.backbone = backbone
            self.head = nn.Sequential(
                nn.Dropout(0.5),
                nn.Linear(in_feat, 256),
                nn.ReLU(inplace=False),
                nn.Dropout(0.3),
                nn.Linear(256, 2),
            )
            self.gradcam_target_layer = backbone.act4
 
        def forward(self, x):
            return self.head(self.backbone(x))
 
    return XceptionNet(base, base.num_features)
 
 
def load_model() -> nn.Module:
    ckpt_path = Path(CFG["checkpoint"])
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {ckpt_path}")
 
    model = build_model()
    ckpt  = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    model = model.to(DEVICE).eval()
    log.info(f"✓ Model dimuat | epoch={ckpt.get('epoch','?')} | val_auc={ckpt.get('val_auc',0):.4f}")
    return model
 
 
# ══════════════════════════════════════════════════════════════
# 4. PREPROCESSING
# ══════════════════════════════════════════════════════════════
 
EVAL_TRANSFORM = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(CFG["norm_mean"], CFG["norm_std"]),
])
 
 
def load_image(img_path: str):
    img_bgr = cv2.imread(img_path)
    if img_bgr is None:
        return None, None
    img_rgb = cv2.resize(
        cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB),
        (CFG["img_size"], CFG["img_size"])
    )
    tensor = EVAL_TRANSFORM(img_rgb).unsqueeze(0).to(DEVICE)
    return img_rgb, tensor
 
 
# ══════════════════════════════════════════════════════════════
# 5. GRAD-CAM
# ══════════════════════════════════════════════════════════════
 
class GradCAM:
    def __init__(self, model: nn.Module):
        self.model  = model
        self._grads = None
        self._acts  = None
        self._hooks = [
            model.gradcam_target_layer.register_forward_hook(
                lambda m, i, o: setattr(self, "_acts", o.detach())
            ),
            model.gradcam_target_layer.register_full_backward_hook(
                lambda m, gi, go: setattr(self, "_grads", go[0].detach())
            ),
        ]
    def __call__(self, tensor: torch.Tensor, class_idx: int = None) -> np.ndarray:
        self.model.zero_grad()
        out = self.model(tensor)
        if class_idx is None:
            class_idx = out.argmax(1).item()
        out[0, class_idx].backward()
 
        cam = (self._grads.mean(dim=[2, 3], keepdim=True) * self._acts).sum(1)
        cam = cam.clamp(min=0).squeeze().cpu().numpy()
 
        if cam.ndim == 0:
            return np.zeros((CFG["img_size"], CFG["img_size"]), dtype=np.float32)
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cv2.resize(cam, (CFG["img_size"], CFG["img_size"])).astype(np.float32)
 
 
# ══════════════════════════════════════════════════════════════
# 6. LIME
# ══════════════════════════════════════════════════════════════
 
def compute_lime(model: nn.Module, img_rgb: np.ndarray) -> np.ndarray:
    def predict_fn(images):
        batch = torch.stack([
            EVAL_TRANSFORM(img.astype(np.uint8)) for img in images
        ]).to(DEVICE)
        with torch.no_grad():
            return F.softmax(model(batch), dim=1).cpu().numpy()
    exp = lime_image.LimeImageExplainer(
        random_state=CFG["random_seed"]
    ).explain_instance(
        img_rgb, predict_fn,
        top_labels=2, hide_color=0,
        num_samples=CFG["lime_num_samples"],
        random_seed=CFG["random_seed"],
    )
    top_label = exp.top_labels[0]
    sal_map   = np.zeros(exp.segments.shape, dtype=np.float32)
    for seg_id, weight in exp.local_exp[top_label]:
        sal_map[exp.segments == seg_id] = weight
 
    if sal_map.max() > sal_map.min():
        sal_map = (sal_map - sal_map.min()) / (sal_map.max() - sal_map.min())
    return sal_map.astype(np.float32)
 
 
# ══════════════════════════════════════════════════════════════
# 7. METRICS
# ══════════════════════════════════════════════════════════════
 
def spearman_corr(a: np.ndarray, b: np.ndarray) -> float:
    r, _ = stats.spearmanr(a.flatten(), b.flatten())
    return float(r)
 
 
def del_ins_auc(
    model: nn.Module,
    tensor: torch.Tensor,
    saliency: np.ndarray,
    mode: str = "deletion",
) -> float:
    H = W    = CFG["img_size"]
    n_px     = H * W
    s_idx    = np.argsort(saliency.flatten())[::-1]
    img_np   = tensor.squeeze(0).cpu().numpy()
    baseline = np.zeros_like(img_np)
    step     = max(1, n_px // CFG["deletion_steps"])
    scores   = []
 
    for s in range(CFG["deletion_steps"] + 1):
        n_mod = min(s * step, n_px)
        midx  = s_idx[:n_mod]
        mod   = img_np.copy()
        for c in range(3):
            ch = mod[c].flatten()
            if mode == "deletion":
                ch[midx] = baseline[c].flatten()[midx]
            else:
                base = baseline[c].flatten().copy()
                base[midx] = img_np[c].flatten()[midx]
                ch = base
            mod[c] = ch.reshape(H, W)
 
        t = torch.tensor(mod, dtype=torch.float32).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            scores.append(F.softmax(model(t), dim=1)[0, 1].item())
 
    return float(sklearn_auc(np.linspace(0, 1, len(scores)), scores))
 
 

 
def run_analysis(model: nn.Module, frame_df: pd.DataFrame):
    gradcam  = GradCAM(model)
    results  = []
    vis_data = []
 
    for _, row in tqdm(frame_df.iterrows(), total=len(frame_df), desc="XceptionNet"):
        img_rgb, tensor = load_image(row["abs_path"])
        if img_rgb is None:
            continue
 
        gc_map   = gradcam(tensor)
        lime_map = compute_lime(model, img_rgb)
        sp       = spearman_corr(gc_map, lime_map)
 
        rec = {
            "model":             CFG["model_name"],
            "filename":          row["filename"],
            "binary_label":      int(row["binary_label"]),
            "manipulation_type": row["manipulation_type"],
            "spearman_corr":     sp,
            "del_auc_gradcam":   del_ins_auc(model, tensor, gc_map,   "deletion"),
            "ins_auc_gradcam":   del_ins_auc(model, tensor, gc_map,   "insertion"),
            "del_auc_lime":      del_ins_auc(model, tensor, lime_map, "deletion"),
            "ins_auc_lime":      del_ins_auc(model, tensor, lime_map, "insertion"),
        }
        results.append(rec)
 
        if len(vis_data) < 4:
            vis_data.append({**rec, "_gc": gc_map, "_lime": lime_map, "_img": img_rgb})
 
        gc.collect()
 
    gradcam.remove_hooks()
    return pd.DataFrame(results), vis_data
 
 

 
def run_stats(results_df: pd.DataFrame):
    sp = results_df["spearman_corr"].dropna()
 
    log.info("\n" + "═"*55)
    log.info(f"  RESULTS — {CFG['model_name']}")
    log.info("═"*55)
    log.info(f"  N Samples       : {len(results_df)}")
    log.info(f"  Spearman mean   : {sp.mean():.4f}")
    log.info(f"  Spearman std    : {sp.std():.4f}")
    log.info(f"  Spearman median : {sp.median():.4f}")
    log.info(f"  Del AUC GradCAM : {results_df['del_auc_gradcam'].mean():.4f}")
    log.info(f"  Ins AUC GradCAM : {results_df['ins_auc_gradcam'].mean():.4f}")
    log.info(f"  Del AUC LIME    : {results_df['del_auc_lime'].mean():.4f}")
    log.info(f"  Ins AUC LIME    : {results_df['ins_auc_lime'].mean():.4f}")

    for lbl, lname in [(0, "real"), (1, "fake")]:
        sub = results_df[results_df["binary_label"] == lbl]["spearman_corr"].dropna()
        log.info(f"  Spearman {lname:<6} : mean={sub.mean():.4f} | std={sub.std():.4f} | n={len(sub)}")
 
    summary = {
        "model":            CFG["model_name"],
        "n":                len(results_df),
        "spearman_mean":    sp.mean(),
        "spearman_std":     sp.std(),
        "spearman_median":  sp.median(),
        "del_auc_gradcam":  results_df["del_auc_gradcam"].mean(),
        "ins_auc_gradcam":  results_df["ins_auc_gradcam"].mean(),
        "del_auc_lime":     results_df["del_auc_lime"].mean(),
        "ins_auc_lime":     results_df["ins_auc_lime"].mean(),
    }
    pd.DataFrame([summary]).to_csv(OUT / "stats_xceptionnet.csv", index=False)
    log.info(f"  Stats disimpan: stats_xceptionnet.csv")
    return summary
 
 

 
def overlay_heatmap(img, hmap, cmap=cv2.COLORMAP_JET):
    h = cv2.cvtColor(
        cv2.applyColorMap((hmap * 255).astype(np.uint8), cmap),
        cv2.COLOR_BGR2RGB,
    )
    return (img * 0.5 + h * 0.5).astype(np.uint8)
 
 
def plot_qualitative(vis_data: list):
    n   = len(vis_data)
    fig = plt.figure(figsize=(12, 4 * n))
    gs  = gridspec.GridSpec(n, 3, figure=fig, hspace=0.35, wspace=0.05)
 
    for i, res in enumerate(vis_data):
        lbl = "REAL" if res["binary_label"] == 0 else "FAKE"
 
        ax0 = fig.add_subplot(gs[i, 0])
        ax0.imshow(res["_img"]); ax0.axis("off")
        ax0.set_title(f"{lbl} | ρ={res['spearman_corr']:.3f}", fontsize=9)
 
        ax1 = fig.add_subplot(gs[i, 1])
        ax1.imshow(overlay_heatmap(res["_img"], res["_gc"], cv2.COLORMAP_JET))
        ax1.axis("off"); ax1.set_title("Grad-CAM", fontsize=9)
 
        ax2 = fig.add_subplot(gs[i, 2])
        ax2.imshow(overlay_heatmap(res["_img"], res["_lime"], cv2.COLORMAP_INFERNO))
        ax2.axis("off"); ax2.set_title("LIME", fontsize=9)
 
    fig.suptitle(f"Qualitative XAI — {CFG['model_name']}", fontsize=12)
    plt.savefig(OUT / "fig_qualitative_xceptionnet.png", dpi=150, bbox_inches="tight")
    plt.close()
    log.info("  Saved: fig_qualitative_xceptionnet.png")
 
 
def plot_spearman(results_df: pd.DataFrame):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
 
    # Histogram real vs fake
    for lbl, color, name in [(0, "#4C72B0", "real"), (1, "#DD8452", "fake")]:
        sub = results_df[results_df["binary_label"] == lbl]["spearman_corr"].dropna()
        axes[0].hist(sub, bins=20, alpha=0.6, color=color,
                     label=f"{name} (n={len(sub)})", density=True)
    axes[0].axvline(0, color="red", linestyle="--", alpha=0.7)
    axes[0].set_title(f"Spearman ρ Distribution — {CFG['model_name']}")
    axes[0].set_xlabel("Spearman ρ"); axes[0].set_ylabel("Density")
    axes[0].legend()
 
    # Violin real vs fake
    results_copy = results_df.copy()
    results_copy["label_name"] = results_copy["binary_label"].map({0: "real", 1: "fake"})
    sns.violinplot(data=results_copy, x="label_name", y="spearman_corr",
                   palette={"real": "#4C72B0", "fake": "#DD8452"},
                   inner="box", ax=axes[1])
    axes[1].axhline(0, color="red", linestyle="--", alpha=0.6)
    axes[1].set_title("Spearman ρ: Real vs Fake")
    axes[1].set_xlabel(""); axes[1].set_ylabel("Spearman ρ")
 
    plt.suptitle(CFG["model_name"], fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / "fig_spearman_xceptionnet.png", dpi=150)
    plt.close()
    log.info("  Saved: fig_spearman_xceptionnet.png")
 
 
def plot_auc(results_df: pd.DataFrame):
    metrics = {
        "Deletion AUC":  ("del_auc_gradcam", "del_auc_lime"),
        "Insertion AUC": ("ins_auc_gradcam", "ins_auc_lime"),
    }
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
 
    for ax, (title, (gc_col, lm_col)) in zip(axes, metrics.items()):
        vals  = [results_df[gc_col].mean(), results_df[lm_col].mean()]
        stds  = [results_df[gc_col].std(),  results_df[lm_col].std()]
        bars  = ax.bar(["Grad-CAM", "LIME"], vals, yerr=stds,
                       color=["#4C72B0", "#DD8452"], capsize=5, alpha=0.85)
        ax.set_ylim(0, 1); ax.set_title(title); ax.set_ylabel("AUC")
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, v + 0.02,
                    f"{v:.3f}", ha="center", fontsize=9)
 
    plt.suptitle(f"Deletion / Insertion AUC — {CFG['model_name']}", fontsize=12)
    plt.tight_layout()
    plt.savefig(OUT / "fig_auc_xceptionnet.png", dpi=150)
    plt.close()
    log.info("  Saved: fig_auc_xceptionnet.png")
 

2026-05-06 14:27:32,387 [INFO] Device : cuda
2026-05-06 14:27:32,388 [INFO] Model  : XceptionNet


In [3]:
def main():
    log.info("═"*55)
    log.info(f"  XAI Evaluation — {CFG['model_name']}")
    log.info("═"*55)
 
    df      = build_dataframe(CFG["test_dir"])
    sample  = sample_balanced(df)
    model   = load_model()
 
    results_df, vis_data = run_analysis(model, sample)
    results_df.to_csv(OUT / "results_xceptionnet.csv", index=False)
 
    summary = run_stats(results_df)
 
    log.info("\nMembuat visualisasi...")
    plot_qualitative(vis_data)
    plot_spearman(results_df)
    plot_auc(results_df)
 
    log.info(f"\n  Output: {OUT.resolve()}")
    log.info("═"*55)
    return results_df, summary
 
 
if __name__ == "__main__":
    main()

2026-05-06 14:27:32,401 [INFO] ═══════════════════════════════════════════════════════
2026-05-06 14:27:32,402 [INFO]   XAI Evaluation — XceptionNet
2026-05-06 14:27:32,403 [INFO] ═══════════════════════════════════════════════════════
2026-05-06 14:27:32,419 [INFO]   'fake' → label=1 | 150 gambar
2026-05-06 14:27:32,433 [INFO]   'real' → label=0 | 150 gambar
2026-05-06 14:27:32,455 [INFO] Total: 300 | Real: 150 | Fake: 150
2026-05-06 14:27:32,461 [INFO] Sampling: 300 gambar (150 real + 150 fake)
2026-05-06 14:27:34,188 [INFO] ✓ Model dimuat | epoch=27 | val_auc=0.9733
XceptionNet:   0%|          | 0/300 [00:00<?, ?it/s]

  0%|          | 0/1000 [00:00<?, ?it/s]

XceptionNet:   0%|          | 1/300 [00:17<1:26:31, 17.36s/it]

  0%|          | 0/1000 [00:00<?, ?it/s]

XceptionNet:   1%|          | 2/300 [00:32<1:20:22, 16.18s/it]

  0%|          | 0/1000 [00:00<?, ?it/s]

... (591 outputs left out)

  0%|          | 0/1000 [00:00<?, ?it/s]

XceptionNet: 100%|█████████▉| 299/300 [1:23:11<00:16, 16.55s/it]

  0%|          | 0/1000 [00:00<?, ?it/s]

XceptionNet: 100%|██████████| 300/300 [1:23:27<00:00, 16.69s/it]
2026-05-06 15:51:01,558 [INFO] 
═══════════════════════════════════════════════════════
2026-05-06 15:51:01,559 [INFO]   RESULTS — XceptionNet
2026-05-06 15:51:01,560 [INFO] ═══════════════════════════════════════════════════════
2026-05-06 15:51:01,561 [INFO]   N Samples       : 300
2026-05-06 15:51:01,561 [INFO]   Spearman mean   : 0.1121
2026-05-06 15:51:01,562 [INFO]   Spearman std    : 0.1905
2026-05-06 15:51:01,563 [INFO]   Spearman median : 0.1090
2026-05-06 15:51:01,564 [INFO]   Del AUC GradCAM : 0.3332
2026-05-06 15:51:01,565 [INFO]   Ins AUC GradCAM : 0.2241
2026-05-06 15:51:01,566 [INFO]   Del AUC LIME    : 0.4941
2026-05-06 15:51:01,567 [INFO]   Ins AUC LIME    : 0.3085
2026-05-06 15:51:01,569 [INFO]   Spearman real   : mean=0.0937 | std=0.1711 | n=150
2026-05-06 15:51:01,570 [INFO]   Spearman fake   : mean=0.1306 | std=0.2071 | n=150
2026-05-06 15:51:01,572 [INFO]   Stats disimpan: stats_xceptionnet.csv
2026-

In [4]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/wesleypeyfendo
/kaggle/input/datasets/wesleypeyfendo/models
/kaggle/input/datasets/wesleypeyfendo/test-images
/kaggle/input/datasets/wesleypeyfendo/test-images/test_images
/kaggle/input/datasets/wesleypeyfendo/test-images/test_images/fake
/kaggle/input/datasets/wesleypeyfendo/test-images/test_images/real
